In [ ]:
import sys; sys.path.insert(0, "..")
from config.settings import DATA_DIR
from src.utils.data_store import load
from src.cleaning.text_cleaner import clean_scopus
from src.analysis.lda_model import build_vectorizer, build_lda, optimize_lda, compute_perplexity

df = load("scopus_devops", DATA_DIR)

# Use Abstract_clean when available; fall back to Title_clean or cleaned Title
abstract_texts = [t for t in df["Abstract_clean"].dropna().tolist() if str(t).strip()]
if len(abstract_texts) < len(df) * 0.1:
    if "Title_clean" in df.columns:
        title_texts = [t for t in df["Title_clean"].dropna().tolist() if str(t).strip()]
    else:
        title_texts = [clean_scopus(str(t)) for t in df["Title"].dropna().tolist() if str(t).strip()]
    texts = abstract_texts + title_texts
    print(f"Only {len(abstract_texts)}/{len(df)} abstracts — using titles as fallback")
else:
    texts = abstract_texts
texts = [t for t in texts if t.strip()]

dtm, vectorizer = build_vectorizer(texts)
print(f"Vectorized {len(texts)} documents, vocab size: {dtm.shape[1]}")

In [ ]:
# Optional: auto-optimize k, alpha, beta via differential evolution (slow)
# best = optimize_lda(dtm, k_min=10, k_max=100, maxiter=5)
# print(best)
# k, alpha, beta = best["k"], best["alpha"], best["beta"]
k, alpha, beta = 20, 0.1, 0.01  # quick demo defaults

In [ ]:
model, doc_topic_matrix = build_lda(dtm, k=k, alpha=alpha, beta=beta, passes=20)
perp = compute_perplexity(model, dtm)
print(f"Perplexity: {perp:.4f}")
print(f"Doc-topic matrix shape: {doc_topic_matrix.shape}")

In [ ]:
from src.analysis.lda_model import get_top_words
top_words = get_top_words(model, vectorizer)
for _, row in top_words.head(5).iterrows():
    print(f"Topic {row['topic_id']}: {row['top_words']}")